In [3]:
import pandas as pd
import numpy as np
import holidays
from datetime import timedelta
import pytz

# Load US holidays to help identify unexpected gaps caused by holidays
us_holidays = holidays.US(years=range(2010, 2027))

In [ ]:
# 1. Load data
file_path = 'nq_final.csv'
df = pd.read_csv(file_path)

# 2. Convert to DateTime and set Timezone
# Databento provides UTC (+00:00). We parse it, then convert to New York time.
df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
df = df.sort_values('datetime')

# Convert to Eastern Time (handles Daylight Saving Time automatically)
df['datetime_et'] = df['datetime'].dt.tz_convert('America/New_York')

# Set as index for easy manipulation
df.set_index('datetime_et', inplace=True)

print(f"Data loaded from {df.index.min()} to {df.index.max()} (Eastern Time)")
print(f"Total rows: {len(df):,}")

Data loaded from 2010-06-06 18:00:00-04:00 to 2026-06-01 19:59:00-04:00 (Eastern Time)
Total rows: 7,136,029


In [5]:
# 1. Calculate the time difference between consecutive rows
df['time_delta'] = df.index.to_series().diff()

# 2. Filter for gaps greater than 1 minute
gaps = df[df['time_delta'] > pd.Timedelta(minutes=1)].copy()

# Add the start time of the gap (the timestamp of the previous row)
gaps['gap_start'] = df.index.to_series().shift(1)[gaps.index]
gaps['gap_end'] = gaps.index

def classify_gap(row):
    start = row['gap_start']
    end = row['gap_end']
    delta = row['time_delta']
    
    start_time = start.time()
    start_dow = start.dayofweek # 0=Mon, 4=Fri, 6=Sun
    
    # Validation 1: Expected Weekend (Friday ~17:00 to Sunday ~18:00)
    # Using >= 16:55 to account for slight vendor variations in final minute bars
    if start_dow == 4 and start.hour >= 16 and start.minute >= 55:
        if end.dayofweek == 6 and end.hour >= 17:
            return 'Expected Weekend'
            
    # Validation 2: Expected Daily Halt (Mon-Thu ~17:00 to ~18:00)
    if start_dow in [0, 1, 2, 3] and start.hour >= 16 and start.minute >= 55:
        # Check if the gap is roughly 1 hour (allowing for 59-65 mins variance)
        if pd.Timedelta(minutes=58) <= delta <= pd.Timedelta(minutes=65):
            return 'Expected Daily Halt'
            
    # Validation 3: Known Holiday Check
    # If the gap starts or ends on a US Holiday, it's a holiday schedule
    if start.date() in us_holidays or end.date() in us_holidays:
        return 'Holiday Schedule (Verify Manually)'
        
    # Validation 4: Illiquid / Zero-Volume Micro Gaps
    # 2 to 5 minute gaps during regular hours are usually just minutes with 0 volume
    if pd.Timedelta(minutes=1) < delta <= pd.Timedelta(minutes=5):
        return 'Micro Gap (Zero Volume)'
        
    # If it passes all the above, it's a genuine unexpected data outage
    return 'UNEXPECTED DATA GAP'

# Apply the classification function
gaps['classification'] = gaps.apply(classify_gap, axis=1)

ValueError: cannot reindex on an axis with duplicate labels

In [ ]:
# 1. Print high-level summary
print("--- GAP ANALYSIS SUMMARY ---")
print(gaps['classification'].value_counts())
print("-" * 28)

# 2. Isolate truly unexpected gaps for manual review
unexpected_gaps = gaps[gaps['classification'] == 'UNEXPECTED DATA GAP']

if len(unexpected_gaps) == 0:
    print("\n✅ SUCCESS: No unexpected large data gaps found in the dataset.")
else:
    print(f"\n⚠️ WARNING: Found {len(unexpected_gaps)} unexpected gaps!")
    print("\nTop 15 Largest Unexpected Gaps:")
    
    # Format a clean dataframe for display
    display_df = unexpected_gaps[['gap_start', 'gap_end', 'time_delta']].copy()
    display_df = display_df.sort_values('time_delta', ascending=False).head(15)
    
    # Format columns for readability
    display_df['gap_start'] = display_df['gap_start'].dt.strftime('%Y-%m-%d %H:%M')
    display_df['gap_end'] = display_df['gap_end'].dt.strftime('%Y-%m-%d %H:%M')
    display_df['duration'] = display_df['time_delta'].astype(str).str.replace('0 days ', '')
    
    display(display_df[['gap_start', 'gap_end', 'duration']])

--- GAP ANALYSIS SUMMARY ---
classification
Micro Gap (Zero Volume)               121660
UNEXPECTED DATA GAP                     7163
Holiday Schedule (Verify Manually)      5119
Expected Daily Halt                     2095
Expected Weekend                         510
Name: count, dtype: int64
----------------------------

⚠️ WARNING: Found 7163 unexpected gaps!

Top 15 Largest Unexpected Gaps:


,gap_start,gap_end,duration
datetime_et,,,
2014-06-15 18:00:00-04:00,2014-06-11 17:14,2014-06-15 18:00,4 days 00:46:00
2014-09-26 07:22:00-04:00,2014-09-22 17:14,2014-09-26 07:22,3 days 14:08:00
2011-12-27 06:00:00-05:00,2011-12-23 16:14,2011-12-27 06:00,3 days 13:46:00
2012-01-03 06:00:00-05:00,2011-12-30 16:14,2012-01-03 06:00,3 days 13:46:00
2020-12-27 18:00:00-05:00,2020-12-24 13:14,2020-12-27 18:00,3 days 04:46:00
2015-12-27 18:00:00-05:00,2015-12-24 13:14,2015-12-27 18:00,3 days 04:46:00
2010-12-26 18:00:00-05:00,2010-12-23 16:14,2010-12-26 18:00,3 days 01:46:00
2011-04-24 18:00:00-04:00,2011-04-21 16:14,2011-04-24 18:00,3 days 01:46:00
2021-12-26 18:00:00-05:00,2021-12-23 16:59,2021-12-26 18:00,3 days 01:01:00
